<a href="https://colab.research.google.com/github/ArghyaRC96/metricguard-ai/blob/main/notebooks/06_baseline_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ MetricGuard AI — Baseline Grounded RAG

## Notebook 06

This notebook combines MetricGuard's production retrieval pipeline with the
Gemini API to generate grounded, structured answers.

Pipeline:

user question
→ dense Qdrant retrieval
→ mandatory Cross-Encoder reranking
→ top-5 evidence
→ Gemini
→ structured answer
→ validated evidence citations
→ confidence / insufficient-evidence fallback

The LLM is not allowed to use ground-truth evaluation files.

Current main LLM:

`gemini-3.7-flash`

In [1]:
from pathlib import Path
import shutil
import subprocess

GITHUB_USERNAME = "ArghyaRC96"

REPO_URL = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/metricguard-ai.git"
)

REPO_DIR = Path(
    "/content/metricguard-ai"
)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git",
        "clone",
        REPO_URL,
        str(REPO_DIR),
    ],
    check=True,
)

print("Repository:", REPO_DIR)

Repository: /content/metricguard-ai


In [2]:
%pip install -q \
    -e "/content/metricguard-ai" \
    google-genai \
    sentence-transformers \
    qdrant-client

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 6.4 MB/s eta 0:00:00
  Building editable for metricguard-ai (pyproject.toml) ... done


In [3]:
import sys

SOURCE_DIR = (
    REPO_DIR
    / "src"
)

if str(SOURCE_DIR) not in sys.path:
    sys.path.append(
        str(SOURCE_DIR)
    )

print(SOURCE_DIR)

/content/metricguard-ai/src


## Load API Key

In [4]:
from google.colab import userdata
from google import genai

# Load Gemini API key securely from Colab Secrets
GEMINI_API_KEY = userdata.get(
    "GEMINI_API_KEY"
)

assert GEMINI_API_KEY, (
    "GEMINI_API_KEY was not found "
    "in Colab Secrets."
)

# Model used by MetricGuard
GEMINI_MODEL = "gemini-3.7-flash"

# Create Gemini client
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini client created.")
print("Model:", GEMINI_MODEL)

✅ Gemini client created.
Model: gemini-3.7-flash


In [5]:
import time

# ---------------------------------------------------------
# GEMINI RATE LIMIT SETTINGS
# ---------------------------------------------------------

# AI Studio currently shows 5 requests per minute.
GEMINI_RPM_LIMIT = 5

# 60 / 5 = 12 seconds.
# Add 2 seconds as a safety buffer.
MIN_GEMINI_INTERVAL = 14

_last_gemini_call = 0.0


# ---------------------------------------------------------
# DEVELOPMENT CACHE
# ---------------------------------------------------------

# Stores already-generated answers during this Colab session.
# If the exact same question is asked again,
# Gemini will NOT be called again.
_gemini_answer_cache = {}


# ---------------------------------------------------------
# RATE LIMIT HELPER
# ---------------------------------------------------------

def wait_for_gemini_slot():
    global _last_gemini_call

    elapsed = (
        time.monotonic()
        - _last_gemini_call
    )

    wait_time = max(
        0,
        MIN_GEMINI_INTERVAL - elapsed,
    )

    if wait_time > 0:
        print(
            f"⏳ Gemini rate-limit safety wait: "
            f"{wait_time:.1f} seconds"
        )

        time.sleep(
            wait_time
        )

    _last_gemini_call = (
        time.monotonic()
    )


print("✅ Gemini rate limiter ready.")
print("✅ Development cache ready.")

✅ Gemini rate limiter ready.
✅ Development cache ready.


In [6]:
wait_for_gemini_slot()

interaction = (
    gemini_client
    .interactions
    .create(
        model=GEMINI_MODEL,
        input=(
            "Reply with exactly: "
            "GEMINI_OK"
        ),
    )
)

print(
    interaction.output_text
)

GEMINI_OK


## Regenerating Chunks

In [7]:
from datetime import date

from metricguard.lineage.enrichment_pipeline import (
    run_full_knowledge_enrichment,
)

run_full_knowledge_enrichment(
    REPO_DIR,
    as_of_date=date(
        2026,
        8,
        18,
    ),
)

METRICGUARD FULL KNOWLEDGE ENRICHMENT REPORT
Parsed documents      : 55
Final chunks          : 167
Metric-aware chunks   : 97
Lineage-aware chunks  : 90
Lineage graph nodes   : 33
Lineage graph edges   : 30
Freshness as-of       : 2026-08-18
Ground truth          : excluded
Embedding readiness   : YES


In [8]:
import json

CHUNKS_PATH = (
    REPO_DIR
    / "data"
    / "processed"
    / "fully_enriched_chunks.jsonl"
)

chunks = []

with CHUNKS_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    for line in file:
        chunks.append(
            json.loads(line)
        )

print(
    "Chunks:",
    len(chunks)
)

Chunks: 167


In [9]:
assert not any(
    "ground_truth"
    in chunk[
        "metadata"
    ].get(
        "source_path",
        "",
    )
    for chunk in chunks
)

print(
    "✅ Ground truth excluded."
)

✅ Ground truth excluded.


In [10]:
from metricguard.retrieval import (
    CrossEncoderReranker,
    DenseRetriever,
    RetrievalPipeline,
    format_final_evidence,
    load_embedding_model,
    load_reranker_model,
    load_retrieval_config,
)

retrieval_config = (
    load_retrieval_config(
        REPO_DIR
    )
)

retrieval_config

RetrievalConfig(candidate_top_k=20, final_top_k=5, reranker_model='cross-encoder/ms-marco-MiniLM-L6-v2', embedding_model='sentence-transformers/all-mpnet-base-v2', collection_name='metricguard_dense_v1', normalize_embeddings=True)

In [11]:
embedding_model = (
    load_embedding_model(
        retrieval_config
        .embedding_model
    )
)

VECTOR_SIZE = (
    embedding_model
    .get_embedding_dimension()
)

print(
    "Vector size:",
    VECTOR_SIZE
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector size: 768


In [12]:
def build_embedding_text(
    chunk: dict,
) -> str:

    metadata = chunk[
        "metadata"
    ]

    parts = [
        (
            "Source type: "
            f"{metadata.get('source_type')}"
        ),
        (
            "Asset type: "
            f"{metadata.get('asset_type')}"
        ),
        (
            "File: "
            f"{metadata.get('file_name')}"
        ),
    ]

    for label, key in [
        ("Metric", "metric_name"),
        (
            "Observed version",
            "observed_version",
        ),
        (
            "Authoritative version",
            "authoritative_version",
        ),
        (
            "Version relation",
            "version_relation",
        ),
        (
            "Freshness",
            "freshness_status",
        ),
    ]:

        value = metadata.get(key)

        if value:
            parts.append(
                f"{label}: {value}"
            )

    return (
        "\n".join(parts)
        + "\n\n"
        + chunk["content"]
    )

In [13]:
embedding_texts = [
    build_embedding_text(
        chunk
    )
    for chunk in chunks
]

embeddings = (
    embedding_model.encode(
        embedding_texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
)

print(
    embeddings.shape
)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

(167, 768)


In [14]:
from qdrant_client import (
    QdrantClient,
    models,
)

qdrant_client = (
    QdrantClient(
        ":memory:"
    )
)

COLLECTION_NAME = (
    retrieval_config
    .collection_name
)

qdrant_client.create_collection(
    collection_name=
        COLLECTION_NAME,
    vectors_config=
        models.VectorParams(
            size=VECTOR_SIZE,
            distance=
                models.Distance.COSINE,
        ),
)

print(
    "✅ Qdrant ready."
)

✅ Qdrant ready.


In [15]:
import uuid

points = []

for chunk, vector in zip(
    chunks,
    embeddings,
):

    point_id = str(
        uuid.uuid5(
            uuid.NAMESPACE_URL,
            chunk["chunk_id"],
        )
    )

    payload = {
        "chunk_id":
            chunk["chunk_id"],
        "content":
            chunk["content"],
        **chunk["metadata"],
    }

    points.append(
        models.PointStruct(
            id=point_id,
            vector=
                vector.tolist(),
            payload=payload,
        )
    )

In [16]:
qdrant_client.upsert(
    collection_name=
        COLLECTION_NAME,
    points=points,
    wait=True,
)

print(
    "Qdrant points:",
    qdrant_client
    .get_collection(
        COLLECTION_NAME
    )
    .points_count,
)

Qdrant points: 167


## Building Production Retriever

In [17]:
production_dense = (
    DenseRetriever(
        client=qdrant_client,
        embedding_model=
            embedding_model,
        collection_name=
            COLLECTION_NAME,
        normalize_embeddings=True,
    )
)

In [18]:
reranker_model = (
    load_reranker_model(
        retrieval_config
        .reranker_model
    )
)

production_reranker = (
    CrossEncoderReranker(
        model=reranker_model,
        batch_size=16,
    )
)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [19]:
production_retrieval = (
    RetrievalPipeline(
        dense_retriever=
            production_dense,
        reranker=
            production_reranker,
        candidate_top_k=
            retrieval_config
            .candidate_top_k,
        final_top_k=
            retrieval_config
            .final_top_k,
    )
)

print(
    "✅ Production retrieval ready."
)

✅ Production retrieval ready.


## Define Gemini's structured answer schema

In [20]:
from typing import Literal

from pydantic import (
    BaseModel,
    Field,
)


class EvidenceUse(BaseModel):

    evidence_id: str

    supports: str


class BaselineRAGAnswer(
    BaseModel
):

    status: Literal[
        "answered",
        "insufficient_evidence",
    ]

    diagnosis: Literal[
        "version_mismatch",
        "stale_definition",
        "intentional_semantic_difference",
        "metric_migration",
        "data_pipeline_issue",
        "insufficient_evidence",
        "other",
    ]

    metric_name: (
        str | None
    ) = None

    answer: str

    key_findings: list[str]

    evidence_used: list[
        EvidenceUse
    ]

    confidence: float = Field(
        ge=0.0,
        le=1.0,
    )

    confidence_reason: str

    missing_evidence: list[str]

In [21]:
def build_evidence_context(
    evidence: list[dict],
) -> str:

    blocks = []

    for index, item in enumerate(
        evidence,
        start=1,
    ):

        evidence_id = (
            f"E{index}"
        )

        block = f"""
[{evidence_id}]

Source: {item.get('source_path')}
File: {item.get('file_name')}
Asset type: {item.get('asset_type')}
Metric: {item.get('metric_name')}
Observed version: {item.get('observed_version')}
Authoritative version: {item.get('authoritative_version')}
Version relation: {item.get('version_relation')}
Freshness: {item.get('freshness_status')}

Direct upstream:
{item.get('direct_upstream')}

All upstream:
{item.get('all_upstream')}

Direct downstream:
{item.get('direct_downstream')}

All downstream:
{item.get('all_downstream')}

CONTENT:
{item.get('content')}
"""

        blocks.append(
            block.strip()
        )

    return "\n\n".join(
        blocks
    )

## Building the MetricGuard baseline prompt

In [22]:
def build_rag_prompt(
    question: str,
    evidence: list[dict],
) -> str:

    evidence_context = (
        build_evidence_context(
            evidence
        )
    )

    return f"""
You are MetricGuard AI, an analytics metric investigation assistant.

Your task is to answer the user's question using ONLY the supplied evidence.

EVIDENCE RULES:

1. Do not use outside knowledge.
2. Do not invent facts, SQL logic, metric definitions, versions, dates, or lineage.
3. Reference evidence only through the supplied evidence IDs such as E1 or E2.
4. A non-current metric version is evidence of a version difference, but it is NOT automatically a defect.
5. A dashboard disagreement can be an intentional semantic difference.
6. Distinguish carefully between:
   - stale definitions,
   - version mismatches,
   - intentional semantic differences,
   - expected metric migrations,
   - genuine pipeline/data problems.
7. If the evidence does not justify a conclusion, set status to "insufficient_evidence".
8. Confidence must reflect the completeness and consistency of the supplied evidence only.
9. Do not cite an evidence ID unless that evidence directly supports the claim.

SUPPLIED EVIDENCE:

{evidence_context}

USER QUESTION:

{question}

Return the structured response required by the schema.
""".strip()

In [23]:
def validate_evidence_ids(
    answer: BaselineRAGAnswer,
    evidence: list[dict],
) -> None:

    allowed = {
        f"E{index}"
        for index in range(
            1,
            len(evidence) + 1,
        )
    }

    used = {
        item.evidence_id
        for item
        in answer.evidence_used
    }

    invalid = (
        used - allowed
    )

    if invalid:
        raise ValueError(
            "Gemini returned invalid "
            f"evidence IDs: {invalid}"
        )

In [24]:
def resolve_sources(
    answer: BaselineRAGAnswer,
    evidence: list[dict],
) -> list[dict]:

    source_map = {
        f"E{index}": item
        for index, item
        in enumerate(
            evidence,
            start=1,
        )
    }

    resolved = []

    for use in (
        answer.evidence_used
    ):

        source = (
            source_map[
                use.evidence_id
            ]
        )

        resolved.append(
            {
                "evidence_id":
                    use.evidence_id,
                "source_path":
                    source.get(
                        "source_path"
                    ),
                "file_name":
                    source.get(
                        "file_name"
                    ),
                "supports":
                    use.supports,
            }
        )

    return resolved

## Creating complete Baseline RAG Function

In [25]:
def ask_metricguard(
    question: str,
):
    # -----------------------------------------------------
    # 1. VALIDATE QUESTION
    # -----------------------------------------------------

    question = question.strip()

    if not question:
        raise ValueError(
            "Question cannot be empty."
        )


    # -----------------------------------------------------
    # 2. CHECK DEVELOPMENT CACHE
    # -----------------------------------------------------

    if question in _gemini_answer_cache:

        print(
            "✅ Using cached Gemini answer."
        )

        return (
            _gemini_answer_cache[
                question
            ]
        )


    # -----------------------------------------------------
    # 3. PRODUCTION RETRIEVAL
    # -----------------------------------------------------

    retrieval_results = (
        production_retrieval
        .retrieve(
            question
        )
    )


    # -----------------------------------------------------
    # 4. FORMAT FINAL TOP-5 EVIDENCE
    # -----------------------------------------------------

    evidence = (
        format_final_evidence(
            retrieval_results
        )
    )


    # -----------------------------------------------------
    # 5. BUILD GROUNDED RAG PROMPT
    # -----------------------------------------------------

    prompt = build_rag_prompt(
        question,
        evidence,
    )


    # -----------------------------------------------------
    # 6. RESPECT GEMINI RPM LIMIT
    # -----------------------------------------------------

    wait_for_gemini_slot()


    # -----------------------------------------------------
    # 7. CALL GEMINI
    # -----------------------------------------------------

    print(
        "🤖 Calling Gemini..."
    )

    interaction = (
        gemini_client
        .interactions
        .create(
            model=GEMINI_MODEL,
            input=prompt,
            generation_config={
                "thinking_level":
                    "medium"
            },
            response_format={
                "type":
                    "text",
                "mime_type":
                    "application/json",
                "schema":
                    BaselineRAGAnswer
                    .model_json_schema(),
            },
        )
    )


    # -----------------------------------------------------
    # 8. PARSE STRUCTURED OUTPUT
    # -----------------------------------------------------

    answer = (
        BaselineRAGAnswer
        .model_validate_json(
            interaction.output_text
        )
    )


    # -----------------------------------------------------
    # 9. VALIDATE EVIDENCE IDs
    # -----------------------------------------------------

    validate_evidence_ids(
        answer,
        evidence,
    )


    # -----------------------------------------------------
    # 10. RESOLVE REAL SOURCE PATHS
    # -----------------------------------------------------

    sources = resolve_sources(
        answer,
        evidence,
    )


    # -----------------------------------------------------
    # 11. BUILD FINAL RESULT
    # -----------------------------------------------------

    result = {
        "question":
            question,

        "answer":
            answer,

        "sources":
            sources,

        "evidence":
            evidence,
    }


    # -----------------------------------------------------
    # 12. CACHE RESULT
    # -----------------------------------------------------

    _gemini_answer_cache[
        question
    ] = result


    print(
        "✅ Gemini answer generated "
        "and cached."
    )

    return result

## Testing Retrieval

In [27]:
revenue_question = (
    "Why does the Executive KPI "
    "Dashboard report different "
    "Net Revenue from the Finance "
    "Revenue Dashboard after "
    "April 1, 2026?"
)

revenue_result = (
    ask_metricguard(
        revenue_question
    )
)

🤖 Calling Gemini...
✅ Gemini answer generated and cached.


In [28]:
print(
    revenue_result[
        "answer"
    ].model_dump_json(
        indent=2
    )
)

{
  "status": "answered",
  "diagnosis": "metric_migration",
  "metric_name": "net_revenue",
  "answer": "The discrepancy occurs because the Finance Revenue Dashboard was migrated to Net Revenue Version 3 (effective April 1, 2026), which deducts posted chargebacks, whereas the Executive KPI Dashboard remains on Net Revenue Version 2 and does not deduct posted chargebacks.",
  "key_findings": [
    "Effective April 1, 2026, the official Net Revenue definition updated to Version 3 to deduct posted chargebacks [E1, E5].",
    "Finance Analytics updated the Finance Revenue Dashboard and Finance Daily mart to Version 3 [E1, E3].",
    "The Executive KPI Dashboard and pipeline have not yet migrated to Version 3 and continue calculating Net Revenue on Version 2 without deducting posted chargebacks [E1, E3, E4].",
    "This version difference causes the Executive KPI Dashboard to report higher Net Revenue values on dates with chargeback activity [E1, E2]."
  ],
  "evidence_used": [
    {
     

In [29]:
print(
    revenue_result[
        "answer"
    ].model_dump_json(
        indent=2
    )
)

{
  "status": "answered",
  "diagnosis": "metric_migration",
  "metric_name": "net_revenue",
  "answer": "The discrepancy occurs because the Finance Revenue Dashboard was migrated to Net Revenue Version 3 (effective April 1, 2026), which deducts posted chargebacks, whereas the Executive KPI Dashboard remains on Net Revenue Version 2 and does not deduct posted chargebacks.",
  "key_findings": [
    "Effective April 1, 2026, the official Net Revenue definition updated to Version 3 to deduct posted chargebacks [E1, E5].",
    "Finance Analytics updated the Finance Revenue Dashboard and Finance Daily mart to Version 3 [E1, E3].",
    "The Executive KPI Dashboard and pipeline have not yet migrated to Version 3 and continue calculating Net Revenue on Version 2 without deducting posted chargebacks [E1, E3, E4].",
    "This version difference causes the Executive KPI Dashboard to report higher Net Revenue values on dates with chargeback activity [E1, E2]."
  ],
  "evidence_used": [
    {
     

In [30]:
revenue_result[
    "sources"
]

[{'evidence_id': 'E1',
  'source_path': 'data/raw/analyst_notes/revenue_v3_migration.md',
  'file_name': 'revenue_v3_migration.md',
  'supports': 'Notes that Finance migrated to Net Revenue v3 (deducting posted chargebacks) while the Executive pipeline remains on v2, causing higher reported values on dates with chargeback activity.'},
 {'evidence_id': 'E2',
  'source_path': 'data/raw/incidents/INC-001.md',
  'file_name': 'INC-001.md',
  'supports': 'Confirms the incident summary that Finance and Executive dashboards report different Net Revenue values after April 1, 2026.'},
 {'evidence_id': 'E3',
  'source_path': 'data/raw/incidents/INC-001.md',
  'file_name': 'INC-001.md',
  'supports': 'Confirms Finance Revenue Dashboard migrated to v3 while the Executive KPI Dashboard calculates Net Revenue without deducting posted chargebacks.'},
 {'evidence_id': 'E4',
  'source_path': 'data/raw/incidents/INC-001.md',
  'file_name': 'INC-001.md',
  'supports': 'Documents the required action to mig

In [31]:
cached_revenue_result = (
    ask_metricguard(
        revenue_question
    )
)

✅ Using cached Gemini answer.


In [32]:
orders_question = (
    "Is the Total Orders disagreement "
    "between Operations and Finance "
    "a data pipeline failure?"
)

orders_result = (
    ask_metricguard(
        orders_question
    )
)

print(
    orders_result[
        "answer"
    ].model_dump_json(
        indent=2
    )
)

🤖 Calling Gemini...
✅ Gemini answer generated and cached.
{
  "status": "answered",
  "diagnosis": "intentional_semantic_difference",
  "metric_name": "total_orders",
  "answer": "No, the Total Orders disagreement between Operations and Finance is not a data pipeline failure. It is an intentional semantic difference: Operations counts all placed orders except cancelled orders to measure operational workload, whereas Finance uses the enterprise definition that only counts successfully paid orders.",
  "key_findings": [
    "The discrepancy is explicitly classified as an intentional and expected semantic difference, not a data pipeline failure (E1, E2).",
    "Operations counts all valid placed orders except cancelled orders for operational workload visibility (E1, E5).",
    "Finance uses the enterprise Total Orders definition introduced in December 2025, which counts only successfully paid orders (E1, E5).",
    "Incident INC-003 tracking this discrepancy between the Operations Dashboa

In [33]:
customer_question = (
    "Why does the Growth and "
    "Marketing Dashboard report "
    "different Active Customers "
    "from the current enterprise "
    "definition?"
)

customer_result = (
    ask_metricguard(
        customer_question
    )
)

print(
    customer_result[
        "answer"
    ].model_dump_json(
        indent=2
    )
)

🤖 Calling Gemini...
✅ Gemini answer generated and cached.
{
  "status": "answered",
  "diagnosis": "version_mismatch",
  "metric_name": "active_customers",
  "answer": "The Growth and Marketing Dashboard reports a different Active Customers count because it is still using the v1 engagement-based definition (counting identified customers with recent website or mobile activity), whereas the enterprise standard was updated on March 1, 2026 to v2 (requiring at least one successfully paid order during the previous 30 days).",
  "key_findings": [
    "The Growth & Marketing Dashboard uses metric version v1 for active_customers, which counts identified customers with recent website or mobile activity.",
    "The enterprise Active Customer definition was updated on March 1, 2026 (v2) to require at least one successfully paid order during the previous 30 days.",
    "Because the Growth dashboard continues to use the previous engagement-based definition rather than v2, it reports a materially hi

In [34]:
unsupported_question = (
    "Why did Northstar Commerce's "
    "European warehouse electricity "
    "cost increase by 17% last month?"
)

unsupported_result = (
    ask_metricguard(
        unsupported_question
    )
)

print(
    unsupported_result[
        "answer"
    ].model_dump_json(
        indent=2
    )
)

🤖 Calling Gemini...
✅ Gemini answer generated and cached.
{
  "status": "insufficient_evidence",
  "diagnosis": "insufficient_evidence",
  "metric_name": null,
  "answer": "There is no evidence available to explain changes in European warehouse electricity costs. The supplied assets only contain documentation and SQL definitions related to orders, transactions, and the Net Revenue metric.",
  "key_findings": [
    "The supplied evidence contains definitions and SQL logic for Net Revenue (v1 and v3), raw source definitions, and executive mart calculations.",
    "No data, documentation, or operational metrics regarding European warehouse facilities, utility bills, or electricity costs exist in the provided evidence."
  ],
  "evidence_used": [],
  "confidence": 0.0,
  "confidence_reason": "None of the provided evidence assets contain information about warehouse operations or electricity costs.",
  "missing_evidence": [
    "Warehouse operational data and facilities expense records for Eu

In [35]:
import yaml

with (
    REPO_DIR
    / "configs"
    / "settings.yaml"
).open(
    "r",
    encoding="utf-8",
) as file:

    settings = (
        yaml.safe_load(file)
    )

MIN_CONFIDENCE = float(
    settings[
        "confidence"
    ][
        "minimum_threshold"
    ]
)

print(
    "Minimum confidence:",
    MIN_CONFIDENCE
)

Minimum confidence: 0.6


In [36]:
def apply_answer_fallback(
    result: dict,
) -> dict:

    answer = result[
        "answer"
    ]

    fallback = (
        answer.status
        == "insufficient_evidence"
        or answer.confidence
        < MIN_CONFIDENCE
    )

    result[
        "fallback_triggered"
    ] = fallback

    if fallback:

        result[
            "display_answer"
        ] = (
            "MetricGuard does not have "
            "enough reliable evidence "
            "to answer this question "
            "with the required confidence."
        )

    else:

        result[
            "display_answer"
        ] = (
            answer.answer
        )

    return result

In [37]:
unsupported_result = (
    apply_answer_fallback(
        unsupported_result
    )
)

print(
    "Fallback:",
    unsupported_result[
        "fallback_triggered"
    ]
)

print(
    unsupported_result[
        "display_answer"
    ]
)

Fallback: True
MetricGuard does not have enough reliable evidence to answer this question with the required confidence.


In [38]:
assert (
    revenue_result[
        "answer"
    ].status
    in {
        "answered",
        "insufficient_evidence",
    }
)

assert not any(
    "ground_truth"
    in source.get(
        "source_path",
        "",
    )
    for source in (
        revenue_result[
            "sources"
        ]
    )
)

assert all(
    source[
        "evidence_id"
    ].startswith("E")
    for source in (
        revenue_result[
            "sources"
        ]
    )
)

print(
    "✅ Baseline Gemini RAG validated."
)

✅ Baseline Gemini RAG validated.


## Phase 7.1 Summary — Baseline Gemini RAG

MetricGuard now generates grounded answers using the Gemini API.

### Main LLM

`gemini-3.7-flash`

### Complete baseline flow

user question
→ production dense retrieval
→ Qdrant top-20 candidates
→ mandatory Cross-Encoder reranking
→ top-5 evidence
→ grounded prompt
→ Gemini structured output
→ Pydantic validation
→ evidence-ID validation
→ deterministic source resolution
→ confidence threshold
→ insufficient-evidence fallback

### Citation architecture

The LLM cannot invent source paths.

Gemini references only evidence IDs such as E1 or E3.

MetricGuard application code resolves those IDs back to the actual retrieval
source metadata.

### Grounding rule

Gemini is instructed to use only retrieved evidence.

Ground-truth evaluation data remains excluded from retrieval and generation.

### Next

Productionize the Gemini adapter and baseline RAG pipeline under
`src/metricguard/`, then add confidence, audit logging, and evaluation.

## Fetching latest production code

In [39]:
import subprocess

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_DIR),
        "pull",
        "origin",
        "main",
    ],
    check=True,
)

print(
    "✅ Latest production code pulled."
)

✅ Latest production code pulled.


In [40]:
%pip install -q -e "/content/metricguard-ai"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metricguard-ai (pyproject.toml) ... done


In [41]:
from metricguard.llm import (
    build_structured_llm,
)

from metricguard.rag import (
    MetricGuardRAG,
)

In [42]:
production_llm = (
    build_structured_llm(
        repo_root=REPO_DIR,
        client=gemini_client,
        minimum_request_interval_seconds=14.0,
    )
)

print(
    "✅ Production Gemini adapter ready."
)

✅ Production Gemini adapter ready.


In [43]:
production_rag = (
    MetricGuardRAG(
        retrieval_pipeline=
            production_retrieval,
        llm=production_llm,
        minimum_confidence=
            MIN_CONFIDENCE,
        cache_enabled=True,
    )
)

print(
    "✅ Production MetricGuard RAG ready."
)

✅ Production MetricGuard RAG ready.


In [44]:
production_result = (
    production_rag.ask(
        revenue_question
    )
)

In [45]:
print(
    production_result
    .answer
    .model_dump_json(
        indent=2
    )
)

{
  "status": "answered",
  "diagnosis": "version_mismatch",
  "metric_name": "net_revenue",
  "answer": "The discrepancy occurs because the Finance Revenue Dashboard was migrated to Net Revenue Version 3 (v3) starting April 1, 2026, which deducts posted chargebacks, whereas the Executive KPI Dashboard remains on Version 2 (v2) and calculates Net Revenue without deducting posted chargebacks.",
  "key_findings": [
    "Beginning April 1, 2026, the official enterprise Net Revenue definition was updated to Version 3 to deduct posted chargebacks [E1, E5].",
    "Finance Analytics migrated the Finance Revenue Dashboard and Finance Daily mart to Net Revenue v3 [E1, E3].",
    "The Executive KPI Dashboard remains on Net Revenue v2, calculating Net Revenue without deducting posted chargebacks and producing higher values [E1, E3].",
    "An action item exists to review and migrate the Executive KPI mart and dashboard configuration to the v3 definition [E1, E4]."
  ],
  "evidence_used": [
    {


In [46]:
for source in (
    production_result.sources
):

    print(
        source.model_dump()
    )

{'evidence_id': 'E1', 'source_path': 'data/raw/analyst_notes/revenue_v3_migration.md', 'file_name': 'revenue_v3_migration.md', 'supports': 'Notes that Finance migrated to Net Revenue v3 (deducting chargebacks) for April 2026 reporting while the Executive dashboard remains on v2.'}
{'evidence_id': 'E2', 'source_path': 'data/raw/incidents/INC-001.md', 'file_name': 'INC-001.md', 'supports': 'Confirms Finance and Executive dashboards report different Net Revenue values for reporting periods after April 1, 2026.'}
{'evidence_id': 'E3', 'source_path': 'data/raw/incidents/INC-001.md', 'file_name': 'INC-001.md', 'supports': 'Confirms Finance migrated to v3 and the Executive KPI Dashboard calculates Net Revenue without deducting posted chargebacks.'}
{'evidence_id': 'E4', 'source_path': 'data/raw/incidents/INC-001.md', 'file_name': 'INC-001.md', 'supports': 'Identifies the required action to migrate the Executive KPI mart and dashboard configuration to the current definition.'}
{'evidence_id': 

In [47]:
print(
    "Fallback:",
    production_result
    .fallback_triggered
)

print(
    "Confidence:",
    production_result
    .answer
    .confidence
)

print(
    "Display answer:"
)

print(
    production_result
    .display_answer
)

Fallback: False
Confidence: 1.0
Display answer:
The discrepancy occurs because the Finance Revenue Dashboard was migrated to Net Revenue Version 3 (v3) starting April 1, 2026, which deducts posted chargebacks, whereas the Executive KPI Dashboard remains on Version 2 (v2) and calculates Net Revenue without deducting posted chargebacks.


In [48]:
cached_result = (
    production_rag.ask(
        revenue_question
    )
)

print(
    "Cache hit:",
    cached_result.cache_hit
)

Cache hit: True


## Phase 7.2 Summary — Production Gemini RAG

The successful baseline RAG prototype has been moved into reusable
production modules.

### LLM layer

`src/metricguard/llm/`

Provides:

- provider-independent structured LLM interface
- Gemini adapter
- LLM configuration loading
- lazy Gemini client construction
- request-spacing protection
- future provider abstraction

### RAG layer

`src/metricguard/rag/`

Provides:

- structured answer schemas
- grounded prompt construction
- deterministic evidence-ID handling
- source resolution
- confidence fallback
- insufficient-evidence behavior
- runtime answer caching
- complete RAG orchestration

### Runtime flow

question
→ production dense retrieval
→ Qdrant top-20
→ mandatory Cross-Encoder reranking
→ top-5 evidence
→ production RAG prompt
→ production Gemini adapter
→ structured Pydantic answer
→ evidence validation
→ source resolution
→ confidence/fallback
→ cache

### Testing

Local tests use fake retrieval and fake Gemini components.

Real production integration is validated in Colab against the Gemini API.

### Next

Build the Agentic MetricGuard workflow using LangGraph.